# Building a RAG (Retrieval-Augmented Generation) System from Scratch

In this notebook we will build a small **Document Q&A system** over a 10-K financial report (PDF).

RAG has 3 big steps:

1. **Ingest** — read the PDF, split it into chunks (here: page by page), and turn each chunk into a vector (embedding).
2. **Retrieve** — when a user asks a question, turn the question into a vector too, and find the most similar chunks stored in our vector database.
3. **Generate** — send the retrieved chunks + the question to an LLM (Groq's `llama-3.3-70b-versatile`), and let it answer using only that context.

We will use:
- `pdfplumber` → read text from the PDF
- `sentence-transformers` → convert text into embeddings (vectors)
- `chromadb` → store embeddings and search them (our vector database)
- `groq` → call the LLM to generate the final answer

Run the cells **in order, top to bottom**.

## Step 0 — Install required libraries

If you are running this in Google Colab, run this cell first (only needed once per session).

In [ ]:
!pip install -q pdfplumber sentence-transformers chromadb groq python-dotenv

## Step 1 — Set the PDF path

Change `PDF_PATH` below if your file has a different name. Make sure the PDF is uploaded to your Colab / working folder before running this cell.

In [ ]:
import pdfplumber
import pandas as pd

PDF_PATH = "_10-K-2025-As-Filed.pdf"   # <-- change this if your file name is different

## Step 2 — Open the PDF

`pdfplumber.open()` loads the PDF so we can read its pages. Printing `pages` just shows the list of page objects (not the text yet) — that's expected.

In [ ]:
pdf = pdfplumber.open(PDF_PATH)
pages = pdf.pages
print(pages)
print("Total pages:", len(pages))

## Step 3 — Look at the text of one page

Let's peek at page 1 to make sure text extraction is working correctly before we process the whole document.

In [ ]:
page_1_text = pages[0].extract_text()
print(page_1_text)

## Step 4 — Load an embedding model

An **embedding model** converts text into a list of numbers (a vector) that captures its meaning. Two pieces of text with similar meaning will have vectors that are close together.

We use `all-MiniLM-L6-v2` — small, fast, and good enough for learning purposes.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

## Step 5 — Try embedding page 1

Let's see what an embedding actually looks like — a vector of numbers, and check its length (should be 384 for this model).

In [ ]:
vector1 = model.encode(page_1_text)
print("Vector length:", len(vector1))
print(vector1[:10], "...")   # just show first 10 numbers

## Step 6 — Create a vector database (ChromaDB)

`chromadb` stores our embeddings on disk (in the `./my_chromadb` folder) so we can search them later.

**Fix:** the original code passed `embedding_function=None`, but that makes Chroma expect *you* to always pass embeddings manually everywhere (fine here since we do that), so we keep it — just be aware this is why we always compute embeddings ourselves before calling `collection.add(...)` or `collection.query(...)`.

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./my_chromadb")

collection = chroma_client.get_or_create_collection(
    name="finance",
    embedding_function=None
)

## Step 7 — Ingest the whole PDF into the vector database

For every page:
1. Extract the text
2. Skip empty pages (scanned images, blank pages, etc.)
3. Create an embedding (`normalize_embeddings=True` makes similarity search more accurate)
4. Store the embedding + the original text + metadata (page number, source file) in ChromaDB

This is the step that actually builds our searchable knowledge base.

In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()
        if not text or not text.strip():
            continue
        text = text.strip()

        embedding = model.encode(
            text,
            normalize_embeddings=True
        ).tolist()

        collection.add(
            ids=[f"page_{page_number}"],
            embeddings=[embedding],
            documents=[text],
            metadatas=[
                {
                    "page": page_number,
                    "source": PDF_PATH
                }
            ]
        )

print("Ingestion complete. Total chunks stored:", collection.count())

## Step 8 — Retrieval: search the vector database

Now the fun part. We take a user's **query**, embed it the same way we embedded the pages, and ask ChromaDB for the `top_k` most similar pages.

This function returns the retrieved page texts joined together — this becomes the "context" we give to the LLM.

In [ ]:
query = "Provide the CONSOLIDATED BALANCE SHEETS"

def retrieve_docs(query, top_k=3):
    q_embedding = model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_embedding, n_results=top_k)
    return "\n\n".join(results["documents"][0])

## Step 9 — Test retrieval

Let's check that retrieval actually finds relevant pages for our query.

**Fix:** the original variable name had a typo (`retreived_text`) — corrected to `retrieved_text` below.

In [ ]:
retrieved_text = retrieve_docs(query)
print("Match found:\n", retrieved_text)

## Step 10 — Load your Groq API key

Create a file named `.env` in the same folder as this notebook with one line:

```
GROQ_API_KEY=your_key_here
```

`load_dotenv()` reads that file and makes the key available to Python via `os.environ`.

In [ ]:
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

## Step 11 — Create the Groq client

This connects to Groq's API using your key, so we can call the LLM.

In [ ]:
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

## Step 12 — Generation: ask the LLM to answer using our retrieved context

This is the "G" in RAG. We send the LLM:
- A **system prompt** telling it to only use the given context (so it doesn't make things up)
- The **context** (the pages we retrieved) and the **question**

The LLM then generates a grounded answer.

In [ ]:
MODEL = "llama-3.3-70b-versatile"

def response(context, query):
    chat_completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a precise technical assistant. Answer questions using only the provided context."
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {query}"
            }
        ]
    )
    return chat_completion.choices[0].message.content

## Step 13 — Put it all together: run the full RAG pipeline

**Fix:** the original notebook defined the `response()` function but never actually called it! Here we call it with our retrieved context and query, and print the final answer.

In [ ]:
final_answer = response(retrieved_text, query)
print("Question:", query)
print("\nAnswer:\n", final_answer)

## Step 14 — Try your own question

Now change the `query` below and re-run retrieval + generation to ask anything about the document.

In [ ]:
my_query = "What was the total revenue reported?"   # <-- try your own question here

my_context = retrieve_docs(my_query)
my_answer = response(my_context, my_query)

print("Question:", my_query)
print("\nAnswer:\n", my_answer)

## Summary

You just built a complete RAG pipeline:

1. **Ingest**: PDF → text (per page) → embeddings → stored in ChromaDB
2. **Retrieve**: question → embedding → similarity search → top matching pages
3. **Generate**: retrieved pages + question → LLM → grounded answer

**Ideas to extend this for practice:**
- Split pages into smaller chunks (e.g. 500 words) for more precise retrieval
- Show the page numbers of the retrieved chunks alongside the answer (for citation)
- Try a different `top_k` and see how the answer quality changes
- Add a simple loop so you can keep asking questions without re-running cells